# Scoring and transforming instances with biomolecular models

In [1]:
import torch
from evedesign.system import System, Protein, SystemInstance, EntityInstance, Mutation
from evedesign.models.esm2 import ESM2
from evedesign.types import DeviceType

DEVICE: DeviceType = "cuda" if torch.cuda.is_available() else "cpu"

/Users/thomashopf/mambaforge/envs/modal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define system

For this example, we use E.coli beta-lactamase, with the signal peptide (1-23) removed from the target sequence.

In [2]:
target_seq = "HPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRVDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEPELNEAIPNDERDTTMPAAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWFIADKSGAGERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW"

system = System(
    Protein(
        id="BLAT_ECOLX", rep=target_seq, first_index=24, sequences=None, structures=None
    )
)

We also set up a corresponding instance for the reference sequence. As we specified the full target sequence on the entity above, we could also use `system.rep_to_instance()` instead of explicitly defining the instance.

In [3]:
target_instance = SystemInstance([
    EntityInstance(rep=target_seq)
])

## Set up model

Here, we use the ESM-2 LLM as an example model.

In [4]:
esm = ESM2(
    model_name="esm2_t33_650M_UR50D"
).build(system)

## Functions for scoring

Note: which of the following functions are available depends on the particular model used

### General instance scoring with score()

This is a very general function for scoring a list of instances, and can be anything from a log-likelihood for a particular sequence to a predicted Tm score from a 3D structure prediction model. The score will be attached to the `score` attribute of the instance, a confidence value to the `confidence` attribute.

In [5]:
scored_instances = esm.score([target_instance])

Some weights of the model checkpoint at facebook/esm2_t33_650M_UR50D were not used when initializing EsmForMaskedLM: ['esm.embeddings.position_embeddings.weight']
- This IS expected if you are initializing EsmForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [6]:
scored_instances[0].score

-50.482994079589844

a### Mutation-based scoring

The following functions are only available for models that support the notion of a mutation relative to a defined target instance.

#### Single mutation scan

This function computes a single mutation matrix relative to a specified target instance (i.e., the self-substitution is assigned a score of 0). The matrix is computed for all positions across all entities by default, unless the `entity `and `positions` attributes are specified.

Note this function may use internal optimizations to speed up the calculation or give more accurate results compared to *score()*, e.g. by computing all substitutions for a position in one model forward pass, so should be used preferentially where possible.

In [7]:
esm.single_mutation_scan(target_instance)

A         C         D         E         F         G  \
entity pos ref                                                               
0      24  H    1.897530 -1.949956  0.672847  0.941789 -0.203164  0.122416   
       25  P    0.347349 -3.463524 -0.257046 -0.694496 -2.036752 -1.053482   
       26  E    0.195137 -3.726057 -0.220572  0.000000 -2.181108 -1.163836   
       27  T    0.306194 -3.197289 -1.087994 -0.931498 -1.438717 -1.104186   
       28  L   -0.157573 -3.555436 -1.383918 -1.250697 -1.449367 -1.626740   
...                  ...       ...       ...       ...       ...       ...   
       282 L   -3.160831 -6.305289 -9.752590 -7.105993 -3.565447 -7.726133   
       283 I    1.248161 -3.298944 -3.866124 -2.901722 -1.974510 -1.133632   
       284 K    0.372354 -5.155786 -0.952658  0.265541 -5.408612 -0.608604   
       285 H    1.217974 -2.370991  1.244300  0.651884 -2.091382  1.244879   
       286 W    0.950500 -1.900166  1.300481  0.678290  1.048050  1.196092   

                       H         I         K         L         M         N  \
entity pos ref                                                               
0      24  H    0.000000  0.453818  1.639335  1.620786  8.837504  0.872523   
       25  P   -1.336630 -0.903465 -0.316085 -0.747004 -2.194591  0.263763   
       26  E   -1.905208 -1.486561 -0.719966 -0.846700 -2.047047 -0.945268   
       27  T   -1.064365 -0.405946 -0.766332  0.482081 -2.156348 -0.775013   
       28  L   -1.711987 -0.783981 -1.285058  0.000000 -2.057038 -1.485333   
...                  ...       ...       ...       ...       ...       ...   
       282 L   -7.326848 -1.164783 -7.230973  0.000000 -2.601101 -8.502551   
       283 I   -3.591177  0.000000 -2.362875 -0.117468 -1.215593 -3.043872   
       284 K   -1.791572 -4.214036  0.000000 -2.941490 -3.861472 -0.627168   
       285 H    0.000000 -1.084223  0.090564 -0.081363 -1.939682  1.038755   
       286 W    0.971447  0.957887  0.466466  2.500653  0.021332  1.273449   

                       P         Q         R         S         T         V  \
entity pos ref                                                               
0      24  H    1.764069  1.522943  1.069281  1.808869  1.258829  1.302791   
       25  P    0.000000 -0.279183 -0.759298  0.944582  0.406040 -1.015208   
       26  E   -0.407207 -0.606882 -1.375408 -0.086674 -0.353024 -0.865834   
       27  T    0.241393 -0.349593 -0.629219  0.079642  0.000000 -0.342355   
       28  L   -0.449963 -1.175781 -0.980554  0.033784 -0.529450 -0.838074   
...                  ...       ...       ...       ...       ...       ...   
       282 L   -8.172620 -6.630079 -6.365013 -5.689997 -4.683598 -1.363851   
       283 I   -3.082389 -2.399626 -2.147208 -0.485048 -0.430317  1.303835   
       284 K   -0.987978 -0.006708 -0.593493  0.050879 -0.776162 -3.003065   
       285 H    0.005784  0.568367  0.337682  1.344925  0.912919 -0.436112   
       286 W    1.821921  1.156183  1.226770  1.849379  0.258510  0.447539   

                       W         Y  
entity pos ref                      
0      24  H   -1.167079 -0.466677  
       25  P   -4.082061 -2.699934  
       26  E   -3.849709 -2.905017  
       27  T   -1.711781 -2.131514  
       28  L   -2.737777 -2.447433  
...                  ...       ...  
       282 L   -5.328467 -5.503359  
       283 I   -3.649039 -3.359537  
       284 K   -5.995267 -4.645043  
       285 H   -3.126285 -1.411851  
       286 W    0.000000  0.832190  

[263 rows x 20 columns]

#### Arbitrary mutant scoring

This function allows to score arbitrary single and higher-order mutants relative to the target instance. Individual mutants are specified as a list of *Mutation* objects.

Note this function may use internal optimizations to speed up the calculation or give more accurate results compared to *score()*, so should be used preferentially where possible.

In [8]:
# single mutant
single_mutant = [Mutation(entity=0, pos=180, ref="M", to="T")]
double_mutant = [Mutation(entity=0, pos=180, ref="M", to="T"), Mutation(entity=0, pos=236, ref="G", to="S"),]

scored_mutant_instances = esm.score_mutants(
    target_instance,
    [
        single_mutant,
        double_mutant,
    ]
)

[inst.score for inst in scored_mutant_instances]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[5.1060211062431335, -1.0658193631097674]

#### Conditional mutation scoring with score_conditional()

This function is typically only of relevance for applications where the conditional likelihood P(x_i | x_\i) needs to be computed, e.g. for Gibbs sampling.

## Transform

This operation transforms instances from one representation level to another. In the case of ESM-2 shown here, it maps sequences to embeddings.

The *transform()* method may also set the *score* attribute of the instance if a score can be computed in the same model pass to make computations more efficient.

In [9]:
instances_transformed = esm.transform([target_instance])

In [10]:
# indices: first instance, first entity
instances_transformed[0][0].embedding

array([[ 9.45182443e-02,  8.84119943e-02,  2.38465648e-02, ...,
        -8.20149183e-02,  1.21000335e-01, -6.58196956e-02],
       [ 1.52301773e-01,  1.07781410e-01, -1.68123171e-02, ...,
        -1.46298438e-01,  1.65676892e-01,  1.86714903e-02],
       [ 1.56093135e-01,  6.56763762e-02, -1.77681074e-03, ...,
         6.84123337e-02,  1.25577629e-01,  2.97634639e-02],
       ...,
       [-1.28520653e-04,  1.62136436e-01,  2.56760214e-02, ...,
        -1.99570388e-01, -2.18329549e-01, -6.81631416e-02],
       [ 6.90786093e-02, -2.48125680e-02, -1.29462764e-01, ...,
        -2.66222972e-02,  5.01280427e-02, -2.78342366e-02],
       [-5.63203879e-02,  9.42330286e-02,  2.08682358e-01, ...,
        -2.98920780e-01, -6.42098263e-02,  1.33362547e-01]], dtype=float32)

In [11]:
instances_transformed[0].score

-50.482994079589844